# Bell Canada Churn Prediction (2-Year Horizon)
**Candidate:** Your Name  
**Role:** Data Scientist (Data Engineering & AI)  
**Objective:** Predict 2-year churn and optimize decision threshold for business cost.

## Business Cost Assumptions
- False Negative (missed churner): **$5,000**
- False Positive (unnecessary retention offer): **$200**

## Notebook Plan
1. Setup & paths  
2. Load + inspect data  
3. Clean & feature engineer  
4. Time-based validation split  
5. Build/train models  
6. Optimize threshold by business cost  
7. Evaluate + export outputs

In [ ]:
# =========================
# 1) Setup
# =========================
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dataclasses import dataclass

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve
)
from sklearn.linear_model import LogisticRegression, SGDClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

FN_COST = 5000
FP_COST = 200

print('Libraries loaded.')

### If using Google Colab, mount Google Drive

In [ ]:
# Uncomment if running in Colab
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# =========================
# 2) Paths (edit once)
# =========================
# Option A (Local Jupyter):
DATA_PATH = '../data/customer_churn_1m.csv'
BASE_DIR = '..'

# Option B (Colab + Drive):
# DATA_PATH = '/content/drive/MyDrive/bell_churn_assessment/data/customer_churn_1m.csv'
# BASE_DIR = '/content/drive/MyDrive/bell_churn_assessment'

OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
METRICS_DIR = os.path.join(OUTPUT_DIR, 'metrics')
PLOTS_DIR = os.path.join(OUTPUT_DIR, 'plots')
MODELS_DIR = os.path.join(OUTPUT_DIR, 'models')

for d in [OUTPUT_DIR, METRICS_DIR, PLOTS_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

TARGET_COL = 'churn'      # change if your target column has different name
DATE_COL = 'signup_date'  # change if your date column has different name

print('DATA_PATH:', DATA_PATH)
print('Output directories ready.')

## 3) Load and Inspect Data

In [ ]:
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
display(df.head())
display(df.dtypes.head(20))

In [ ]:
def optimize_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    for col in df.select_dtypes(include=['object']).columns:
        ratio = df[col].nunique(dropna=True) / max(len(df), 1)
        if ratio < 0.5:
            df[col] = df[col].astype('category')
    return df

df = optimize_dtypes(df)
print('Dtype optimization done.')

In [ ]:
print('Target distribution (%):')
display((df[TARGET_COL].value_counts(normalize=True) * 100).rename('percent'))

missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
print('\nTop missing columns (%):')
display(missing_pct.head(15))

## 4) Feature Engineering and Outlier Handling

In [ ]:
def add_date_features(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
        df['signup_year'] = df[date_col].dt.year
        df['signup_month'] = df[date_col].dt.month
        df['signup_quarter'] = df[date_col].dt.quarter
        df['signup_dayofweek'] = df[date_col].dt.dayofweek
    return df

def cap_outliers_iqr(df: pd.DataFrame, numeric_cols: list, iqr_mult: float = 1.5) -> pd.DataFrame:
    q1 = df[numeric_cols].quantile(0.25)
    q3 = df[numeric_cols].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - iqr_mult * iqr
    upper = q3 + iqr_mult * iqr
    df[numeric_cols] = df[numeric_cols].clip(lower=lower, upper=upper, axis=1)
    return df

df = add_date_features(df, DATE_COL)

num_cols_all = df.select_dtypes(include=[np.number]).columns.tolist()
if TARGET_COL in num_cols_all:
    num_cols_all.remove(TARGET_COL)

df = cap_outliers_iqr(df, num_cols_all, iqr_mult=1.5)
print('Feature engineering + outlier capping complete.')

## 5) Time-Based Split (Validation Strategy)
Train on earlier records, validate on later records to mimic production and reduce temporal leakage.

In [ ]:
# Sort by signup date for time split
df = df.sort_values(DATE_COL).reset_index(drop=True)

# Drop raw date after sorting
df_model = df.drop(columns=[DATE_COL])

split_idx = int(len(df_model) * 0.8)
train_df = df_model.iloc[:split_idx].copy()
val_df = df_model.iloc[split_idx:].copy()

X_train = train_df.drop(columns=[TARGET_COL])
y_train = train_df[TARGET_COL].astype(int)
X_val = val_df.drop(columns=[TARGET_COL])
y_val = val_df[TARGET_COL].astype(int)

print('Train:', X_train.shape, y_train.shape)
print('Val  :', X_val.shape, y_val.shape)

In [ ]:
# Optional memory safety step
num_cols_tmp = X_train.select_dtypes(include=[np.number]).columns.tolist()
for c in num_cols_tmp:
    X_train[c] = X_train[c].astype('float32')
    X_val[c] = X_val[c].astype('float32')

print('Numeric columns cast to float32.')

## 6) Preprocess + Train Models
Using sparse one-hot encoding to avoid memory crashes.

In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))  # sparse
])

preprocess = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols)
])

logit_model = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

sgd_model = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', SGDClassifier(
        loss='log_loss',
        class_weight='balanced',
        random_state=RANDOM_STATE
    ))
])

print('Pipelines defined.')

In [ ]:
logit_model.fit(X_train, y_train)
sgd_model.fit(X_train, y_train)

val_prob_logit = logit_model.predict_proba(X_val)[:, 1]

# SGDClassifier: convert decision score to probability-like value
val_score_sgd = sgd_model.decision_function(X_val)
val_prob_sgd = 1 / (1 + np.exp(-val_score_sgd))

def model_metrics(y_true, y_prob):
    return {
        'ROC_AUC': roc_auc_score(y_true, y_prob),
        'PR_AUC': average_precision_score(y_true, y_prob)
    }

m_logit = model_metrics(y_val, val_prob_logit)
m_sgd = model_metrics(y_val, val_prob_sgd)

metrics_df = pd.DataFrame([m_logit, m_sgd], index=['Logistic', 'SGD_LogLoss'])
display(metrics_df)

if m_logit['PR_AUC'] >= m_sgd['PR_AUC']:
    best_name, best_model, best_prob = 'Logistic', logit_model, val_prob_logit
else:
    best_name, best_model, best_prob = 'SGD_LogLoss', sgd_model, val_prob_sgd

print('Selected model:', best_name)

## 7) Threshold Optimization for Business Cost

In [ ]:
@dataclass
class CostResult:
    threshold: float
    tn: int
    fp: int
    fn: int
    tp: int
    total_cost: float
    precision: float
    recall: float

def evaluate_at_threshold(y_true, y_prob, threshold, fn_cost=FN_COST, fp_cost=FP_COST):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    total_cost = fn * fn_cost + fp * fp_cost
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return CostResult(threshold, tn, fp, fn, tp, total_cost, precision, recall)

def find_optimal_threshold(y_true, y_prob, step=0.01):
    rows = []
    for t in np.arange(step, 1.0, step):
        r = evaluate_at_threshold(y_true, y_prob, t)
        rows.append({
            'threshold': r.threshold,
            'tn': r.tn, 'fp': r.fp, 'fn': r.fn, 'tp': r.tp,
            'precision': r.precision, 'recall': r.recall,
            'total_cost': r.total_cost
        })
    table = pd.DataFrame(rows).sort_values('total_cost', ascending=True).reset_index(drop=True)
    return table, table.loc[0, 'threshold']

cost_table, best_threshold = find_optimal_threshold(y_val.values, best_prob, step=0.01)
default_eval = evaluate_at_threshold(y_val.values, best_prob, 0.50)
optimal_eval = evaluate_at_threshold(y_val.values, best_prob, best_threshold)

print(f'Best threshold: {best_threshold:.2f}')
print(f'Cost at threshold 0.50: ${default_eval.total_cost:,.0f}')
print(f'Cost at threshold {best_threshold:.2f}: ${optimal_eval.total_cost:,.0f}')
print(f'Estimated savings: ${default_eval.total_cost - optimal_eval.total_cost:,.0f}')

display(cost_table.head(10))

## 8) Final Evaluation at Optimal Threshold

In [ ]:
y_pred_opt = (best_prob >= best_threshold).astype(int)

print('Confusion Matrix (Optimal Threshold):')
print(confusion_matrix(y_val, y_pred_opt))

print('\nClassification Report (Optimal Threshold):')
print(classification_report(y_val, y_pred_opt, digits=4))

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_val, best_prob)
plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, label=f'{best_name} ROC')
plt.plot([0,1], [0,1], '--')
plt.title('ROC Curve')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'roc_curve.png'), dpi=150)
plt.show()

# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_val, best_prob)
plt.figure(figsize=(6,4))
plt.plot(recall, precision, label=f'{best_name} PR')
plt.title('Precision-Recall Curve')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'pr_curve.png'), dpi=150)
plt.show()

# Cost vs Threshold
plt.figure(figsize=(7,4))
plt.plot(cost_table['threshold'], cost_table['total_cost'])
plt.axvline(best_threshold, color='red', linestyle='--', label=f'Best={best_threshold:.2f}')
plt.title('Business Cost vs Decision Threshold')
plt.xlabel('Threshold')
plt.ylabel('Total Cost ($)')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'cost_vs_threshold.png'), dpi=150)
plt.show()

## 9) Save Output Tables

In [ ]:
metrics_df.to_csv(os.path.join(METRICS_DIR, 'model_probability_metrics.csv'), index=True)
cost_table.to_csv(os.path.join(METRICS_DIR, 'threshold_cost_table.csv'), index=False)

summary = pd.DataFrame([{
    'selected_model': best_name,
    'optimal_threshold': float(best_threshold),
    'default_cost': float(default_eval.total_cost),
    'optimal_cost': float(optimal_eval.total_cost),
    'estimated_savings': float(default_eval.total_cost - optimal_eval.total_cost),
    'fn_cost': FN_COST,
    'fp_cost': FP_COST
}])
summary.to_csv(os.path.join(METRICS_DIR, 'business_summary.csv'), index=False)

print('Saved:')
print('-', os.path.join(METRICS_DIR, 'model_probability_metrics.csv'))
print('-', os.path.join(METRICS_DIR, 'threshold_cost_table.csv'))
print('-', os.path.join(METRICS_DIR, 'business_summary.csv'))

## Task 2 Answer (Validation Strategy)
I used a time-based holdout split rather than random split. Since churn prediction is deployed on future customers, training on earlier signups and validating on later signups better reflects production behavior and reduces temporal leakage risk.

## Task 4 Answer (GCP Architecture for 100M rows)
At 100M rows, I would ingest and store raw data in **Cloud Storage**, perform large-scale transformation using **Dataflow** (or SQL-first transforms in **BigQuery**), and maintain curated feature tables in **BigQuery**. I would train models on **Vertex AI** (custom training or AutoML tabular depending on constraints), with experiment tracking and model registry in Vertex AI. Batch scoring would run via Vertex pipelines and write outputs back to BigQuery. Real-time serving (if needed) would use a deployed Vertex endpoint, while orchestration would be handled by **Cloud Composer** or **Vertex Pipelines**. Monitoring would include model performance drift, data quality checks, and business KPI tracking (retention uplift and cost per saved customer).

## Task 5 Prep (Executive Slide Inputs)
Use these outputs in your one-slide summary:
- Selected model name
- Optimal threshold
- Cost at default threshold (0.50)
- Cost at optimal threshold
- Estimated savings
- Recommended production action: score weekly/monthly and target only customers above optimized threshold